# 01 · Watermark detection
Inspect SynthID detection results for a run. The p-value is P(score ≥ observed | not watermarked with this key). It is not the probability that a text is watermarked.

In [ ]:
from pathlib import Path
import json
import pandas as pd

RESULTS = Path("../results")
# Pick a run explicitly, or use the most recent one that has trials.parquet.
RUN_ID = None
runs = sorted(p for p in RESULTS.iterdir() if (p / "trials.parquet").exists())
RUN = RESULTS / RUN_ID if RUN_ID else runs[-1]
print(RUN.name)

In [ ]:
det = pd.read_parquet(RUN / "tables" / "detections.parquet")
det[["doc_id", "input_kind", "watermark_enabled", "p_value", "watermark_detected", "mean_g", "n_scored", "n_tokens"]]

In [ ]:
from wm_eval.analysis.statistics import detection_rates
pd.DataFrame({k: {"TPR": v["tpr"]["rate"], "TPR CI": v["tpr"]["ci95"], "FPR": v["fpr"]["rate"], "FPR CI": v["fpr"]["ci95"]}
              for k, v in detection_rates(det).items()}).T

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3))
for wm, g in det[det.input_kind == "token_ids"].groupby("watermark_enabled"):
    ax.scatter(g.n_tokens, -np.log10(g.p_value.clip(lower=1e-300)), label=f"watermarked={wm}")
ax.axhline(-np.log10(det.threshold.iloc[0]), ls="--", color="grey")
ax.set_xlabel("generated tokens"); ax.set_ylabel("-log10 p"); ax.legend()

## Detect any text
This uses the generator's own tokenizer and needs `WM_SECRET` in the environment (or `.env` in the project root).

In [ ]:
import os
from wm_eval.config import load_config, load_dotenv
from wm_eval.generation.models import load_tokenizer
from wm_eval.watermark.detector import detect_text
load_dotenv("../.env")
cfg = load_config("../configs/experiments/smoke.yaml")
spec = cfg.generators["qwen3.5-2b"]
doc = json.loads(next((RUN / "artifacts" / "generated").glob("*__wm.json")).read_text())
detect_text("example", doc["text"], load_tokenizer(spec), cfg.watermark, spec.hf_id)